In [1]:
from platform import python_version
print(python_version())

3.11.11


In [2]:
!echo $CONDA_DEFAULT_ENV

pytorch_env


### Calculating all possible Enrichment Analysis
  - for each LFC/FDR cutoff one calculates the Enrichment Analysis
  - We used Enricher python API
     - Reactome (2022)
     - Bioplanet (2019)
     - WikiPathways (2021 Human)
     - KEGG (2021 Human)
     - GO Biological Process (2021)
     - MSigDB Hallmark (2020)
   
### For each enriched pathways one calculates:
  - DEGs in the pathway
  - DEGs not in the pathway
  - TOI1, 2, 3, 4

In [3]:
import os, sys, pickle, yaml
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

from Basic import *
from enricher_lib import *
# from config_lib import *

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [4]:
root_chibe = dic_yml['root_chibe']
root_colab = dic_yml['root_colab']
root0 = dic_yml['root0']

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

disease = dic_yml['disease']
gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

abs_lfc_cutoff_inf = dic_yml['abs_lfc_cutoff_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
num_min_degs_for_ptw_enr = dic_yml['num_min_degs_for_ptw_enr']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_index = dic_yml['saturation_lfc_index']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(project, s_project, case_list, root0)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1
abs_lfc_cutoff, fdr_lfc_cutoff, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={abs_lfc_cutoff:.3f}; fdr={fdr_lfc_cutoff:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'Medulloblastoma microarray study', s_project 'medulloblastoma'
G/P LFC cutoffs: lfc=1.000; fdr=0.050
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [5]:
num_of_genes_list

[3]

In [6]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, num_min_degs_for_ptw_enr=num_min_degs_for_ptw_enr,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          root_colab=root_colab, root_chibe=root_chibe, 
          abs_lfc_cutoff_inf=abs_lfc_cutoff_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_index=saturation_lfc_index, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, abs_lfc_cutoff=1, fdr_lfc_cutoff=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
enr.echo_parameters()

Start opening tables ....
Building synonym dictionary ...

$$$ Removed: ./figures/
Start opening tables ....
Building synonym dictionary ...

>>> WNT
>>> case WNT
	DEGs 5860
		Up (#2695)
		Dw (#3165)

Up-regulated per biotype
                               biotype     n
0                                  LNC   654
1                                  TEC     1
2                                  UNK    48
3                            hasSymbol   240
4                               lncRNA    56
5                                ncRNA    76
6                 processed_pseudogene     7
7                       protein_coding  1591
8                               scaRNA     4
9                                snRNA     1
10                              snoRNA     9
11      transcribed_unitary_pseudogene     2
12  transcribed_unprocessed_pseudogene     5
13              unprocessed_pseudogene     1

Down-regulated per biotype
                               biotype     n
0                         

In [7]:
enr.abs_lfc_cutoff_inf, abs_lfc_cutoff_inf

(0.4, 0.4)

In [8]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

27223


,entrezid,symbol,geneid,name,synonyms,other_names,ec_enzyme,refseq_gen,refseq_prot,refseq_rna,...,acessions_rna,acessions_trans,map_location,dic_panther,ortholog,dic_uniprot,swissprot,wikipedia,refseq_gen,refseq_rna
0,ENSG00000121410,A1BG,1,alpha-1-B glycoprotein,"['A1B', 'ABG', 'GAB', 'HYST2477']","['HEL-S-163pA', 'alpha-1B-glycoprotein', 'epididymis secretory sperm binding...",NaN,NaN,NP_570602.2,NaN,...,"['AA484435.1', 'AB073611.1', 'AF414429.1', 'AI022193.1', 'AK055885.1', 'AK05...","[{'protein': 'AAH35719.1', 'rna': 'BC035719.1'}, {'protein': 'BAG51591.1', '...",19q13.43,"{'HGNC': '5', '_license': 'http://pantherdb.org/tou.jsp', 'ortholog': [{'MGI...","[{'MGI': '2152878', 'ortholog_type': 'LDO', 'panther_family': 'PTHR11738', '...","{'Swiss-Prot': 'P04217', 'TrEMBL': ['M0R009', 'V9HWD8']}",P04217,{'url_stub': 'A1BG (gene)'},"['NC_000019.10', 'NC_060943.1']",NM_130786.4
1,ENSG00000268895,A1BG-AS1,503538,A1BG antisense RNA 1,"['A1BG-AS', 'A1BGAS', 'NCRNA00181']","['A1BG antisense RNA (non-protein coding)', 'A1BG antisense RNA 1 (non-prote...",NaN,NaN,NaN,NaN,...,"['AK027222.1', 'BC006144.1', 'BC040926.1', 'NR_015380.2']",NaN,19q13.43,NaN,NaN,NaN,NaN,NaN,"['NC_000019.10', 'NC_060943.1']",NR_015380.2


In [9]:
lista = [x for x in os.listdir(enr.root_result) if 'medulloblastoma_DEG_' in x and not '~lock' in x]
lista.sort()
print(len(lista))
lista[:3]

63


['medulloblastoma_DEG_DW_IG_V_gene_LFC_WNT_x_CTRL_not_normalized.txt',
 'medulloblastoma_DEG_DW_LNC_LFC_G4_x_CTRL_not_normalized.txt',
 'medulloblastoma_DEG_DW_LNC_LFC_WNT_x_CTRL_not_normalized.txt']

In [10]:
files = [x for x in os.listdir(enr.root_enrichment) if 'Reactome_' in x and not '~lock' in x and '_WNT_' in x]
print(len(files))
files[:2]

7001


['enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.725_fdr_0.230_pathway_pval_0.050_fdr_0.600_num_genes_3.tsv',
 'enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.625_fdr_0.270.tsv']

### Summary of cases - below on can see the enriched tables for different databases

In [11]:
print("")

for case in case_list:
    enr.open_case(case, verbose=False)
    enr.echo_parameters()
    print("\n------------------\n\n")


Enriched Analysis table for WNT does not exist: '../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv'
For case 0 'WNT' ('WNT'), there are 5860/3996 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.900; FDR=1.000
	5860/3996 DEGs/ensembl.
		Up 2695/1677 DEGs/ensembl.
		Dw 3165/2319 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05
No enrichment analysis was calculated.

------------------


Enriched Analysis table for G4 does not exist: '../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_G4_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv'
For case 1 'G4' ('Group 4'), there are 6702/4178 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.900; FDR=1.000
	67

### Cutoffs and Results

In [12]:
for case in case_list:
    ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)

    print(f"For {case}")
    print(f"\tLFC cutoffs: lfc={enr.abs_lfc_cutoff:.3f}; fdr={enr.fdr_lfc_cutoff} #{enr.s_deg_dap}s = {len(degs)}")
    print(f"\tPathway cutoffs: fdr={enr.pathway_fdr_cutoff:.3f}; num of genes={enr.num_of_genes_cutoff}, #Pathways = {enr.n_pathways}, #{enr.s_deg_dap}s in pathwyas = {enr.n_degs_in_pathways}\n")


Enriched Analysis table for WNT does not exist: '../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv'
For WNT
	LFC cutoffs: lfc=0.900; fdr=1 #DEGs = 5860
	Pathway cutoffs: fdr=0.050; num of genes=0.05, #Pathways = 0, #DEGs in pathwyas = 0

Enriched Analysis table for G4 does not exist: '../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_G4_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv'
For G4
	LFC cutoffs: lfc=0.900; fdr=1 #DEGs = 6702
	Pathway cutoffs: fdr=0.050; num of genes=0.05, #Pathways = 0, #DEGs in pathwyas = 0



In [13]:
print(">>> fdr_lfc_cutoff", enr.fdr_lfc_cutoff)

try:
    dflfc = enr.dflfc_ori[(enr.dflfc_ori.fdr < enr.fdr_lfc_cutoff)]
    print(len(dflfc))
except:
    dflfc = pd.DataFrame()

dflfc.head(3)

>>> fdr_lfc_cutoff 1
28232


,probe,symbol,symbol_prev,symb_or_syn,biotype,_type,lfc,abs_lfc,pval,fdr,...,ensembl_id,ensembl_transc_id,geneid,cytoband,symbol_pipe,seqname,start,end,go_id,seq
0,A_19_P00315452,lncRNA6062,NaN,NaN,LNC,LNC,0.093,0.093,0.798,0.939,...,NaN,NaN,NaN,hs|18q21.33,NaN,18,5.927e+07,5.927e+07,NaN,GACTGACGACAAGAAGGCTCTTCGTCCCTGTTTTTGAAATAAACATGGGTATAATCTGAA
1,A_19_P00315469,lncRNA5726,NaN,NaN,LNC,LNC,0.130,0.130,0.609,0.831,...,NaN,NaN,NaN,hs|3q13.11,NaN,3,1.070e+08,1.070e+08,NaN,GCAAATCGGAATCCACGGAAGAAAATTCACCTCTTAAGGATCCAGTCCGGAAAGGGATGG
2,A_19_P00315473,lncRNA5647,NaN,NaN,LNC,LNC,0.139,0.139,0.579,0.812,...,NaN,NaN,NaN,hs|3q25.31,NaN,3,1.568e+08,1.568e+08,NaN,ACTGAACAGAACAGGCAGGAGGTATTTTCTTCTGAAGAGCTGTCAAGAGCCAATACAGCA


In [14]:
# df2 = enr.dflfc_ori[ (enr.dflfc_ori.symbol == 'IGHA2') | (enr.dflfc_ori.symbol == 'A2M')]
# df2

In [15]:
fname_final_ori, fname_ori, title = enr.set_lfc_names()
fname_final_ori, title

('medulloblastoma_final_LFC_G4_x_CTRL_not_normalized.tsv',
 'G4 (not_normalized)')

In [16]:
enr.set_enrichment_name()

('enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_G4_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv',
 'enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_G4_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000_pathway_pval_0.050_fdr_0.050_num_genes_0.05.tsv')

In [17]:
enr.get_best_ptw_cutoff_biopax(verbose=True)
# self.pathway_pval_cutoff, self.pathway_fdr_cutoff, self.num_of_genes_cutoff,

Best parameter file for Pathways does not exist ../../colaboracoes/medulloblastoma/projects/medulloblastoma/config/best_ptw_cutoffs_Medulloblastoma_microarray_study.tsv
Could not find best params for G4 not_normalized 0
Houston we have a problem: No best parameter file for Pathways was found.
>>> run: pubmed_taubate_new05_best_cutoffs_sim_and_save_config.ipynb
>>> Reactome_Pathways_2024


### Testing EnrichR API 

In [18]:
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
ret, len(degs), enr.n_degs, enr.n_degs_ensembl

Enriched Analysis table for WNT does not exist: '../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv'


(True, 5860, 5860, 3996)

In [19]:
# dfdegs.columns

In [20]:
print(len(dfdegs))
dfdegs.head(3)

5860


,probe,symbol,symbol_prev,symb_or_syn,biotype,_type,lfc,abs_lfc,pval,fdr,...,ensembl_id,ensembl_transc_id,geneid,cytoband,symbol_pipe,seqname,start,end,go_id,seq
0,A_33_P3317618,SYN2,SYN2,SYN2,protein_coding,hasSymbol,-6.108,6.108,7.600e-07,0.015,...,ENSG00000157152,ENST00000432424,6854.0,hs|3p25.2,SYN2,3,1.223e+07,1.223e+07,GO:0005524(ATP binding)|GO:0007268(synaptic transmission)|GO:0007269(neurotr...,GATTGAAGTACCTCTCTTTTGTGACTCTTGTACAGCTTAATGTGCAATAAAGGAAAAGTT
1,A_23_P210164,HOXD8,HOXD8,HOXD8,protein_coding,hasSymbol,5.928,5.928,7.329e-07,0.015,...,ENSG00000175879,ENST00000544999,3234.0,hs|2q31.1,HOXD8,2,1.770e+08,1.770e+08,GO:0003700(sequence-specific DNA binding transcription factor activity)|GO:0...,GGAATTTCTTTTTAACCCCTATCTGACCAGGAAAAGAAGAATCGAGGTTTCCCACGCCCT
2,A_23_P37127,FOXA1,FOXA1,FOXA1,protein_coding,hasSymbol,6.132,6.132,1.343e-06,0.018,...,ENSG00000129514,ENST00000250448,3169.0,hs|14q21.1,FOXA1,14,3.806e+07,3.806e+07,GO:0000122(negative regulation of transcription from RNA polymerase II promo...,ATTACAATGTTGGAGGAGAGATAAGTTATAGGGAGCTGGATTTCAAAACGTGGTCCAAGA


### Biotypes

https://www.ensembl.org/info/genome/genebuild/biotypes.html

  - TEC (To be Experimentally Confirmed)


In [21]:
dfdegs_ensembl = dfdegs[ (~pd.isnull(dfdegs.ensembl_id)) & (dfdegs.biotype != 'TEC')].copy()
cols = ['probe', 'symbol', 'symbol_prev', 'symb_or_syn', 'biotype', '_type',
       'lfc', 'abs_lfc', 'fdr', 'description',
       'desc_gff', 'description_prev', 'accession', 'ensembl_id',
       'ensembl_transc_id', 'geneid', 'cytoband', 'symbol_pipe', ]
print(len(dfdegs), len(enr.dflfc), len(dfdegs_ensembl), len(enr.dfdegs_ensembl))
dfdegs_ensembl[cols].head()

5860 5860 3995 3996


,probe,symbol,symbol_prev,symb_or_syn,biotype,_type,lfc,abs_lfc,fdr,description,desc_gff,description_prev,accession,ensembl_id,ensembl_transc_id,geneid,cytoband,symbol_pipe
0,A_33_P3317618,SYN2,SYN2,SYN2,protein_coding,hasSymbol,-6.108,6.108,0.015,"Homo sapiens synapsin II (SYN2), transcript variant IIb, mRNA [NM_003178]",synapsin II [Source:HGNC Symbol%3BAcc:HGNC:11495],synapsin II,NM_003178,ENSG00000157152,ENST00000432424,6854.0,hs|3p25.2,SYN2
1,A_23_P210164,HOXD8,HOXD8,HOXD8,protein_coding,hasSymbol,5.928,5.928,0.015,"Homo sapiens homeobox D8 (HOXD8), transcript variant 1, mRNA [NM_019558]",homeobox D8 [Source:HGNC Symbol%3BAcc:HGNC:5139],homeobox D8,NM_019558,ENSG00000175879,ENST00000544999,3234.0,hs|2q31.1,HOXD8
2,A_23_P37127,FOXA1,FOXA1,FOXA1,protein_coding,hasSymbol,6.132,6.132,0.018,"Homo sapiens forkhead box A1 (FOXA1), mRNA [NM_004496]",forkhead box A1 [Source:HGNC Symbol%3BAcc:HGNC:5021],forkhead box A1,NM_004496,ENSG00000129514,ENST00000250448,3169.0,hs|14q21.1,FOXA1
3,A_23_P39550,TMEM163,TMEM163,TMEM163,protein_coding,hasSymbol,-3.229,3.229,0.020,"Homo sapiens transmembrane protein 163 (TMEM163), mRNA [NM_030923]",transmembrane protein 163 [Source:HGNC Symbol%3BAcc:HGNC:25380],transmembrane protein 163,NM_030923,ENSG00000152128,ENST00000476823,81615.0,hs|2q21.3,TMEM163
4,A_33_P3379606,RGS7BP,RGS7BP,RGS7BP,protein_coding,hasSymbol,-5.518,5.518,0.020,"Homo sapiens regulator of G-protein signaling 7 binding protein (RGS7BP), mR...",regulator of G protein signaling 7 binding protein [Source:HGNC Symbol%3BAcc...,regulator of G protein signaling 7 binding protein,NM_001029875,ENSG00000186479,ENST00000334025,401190.0,hs|5q12.3,RGS7BP


In [22]:
np.unique(dfdegs_ensembl.biotype)

array(['lncRNA', 'processed_pseudogene', 'protein_coding', 'scaRNA',
       'snRNA', 'snoRNA', 'transcribed_processed_pseudogene',
       'transcribed_unitary_pseudogene',
       'transcribed_unprocessed_pseudogene', 'unprocessed_pseudogene'],
      dtype=object)

In [23]:
len(enr.degs), len(enr.degs_ensembl)

(5860, 3996)

In [24]:
enr.n_degs, enr.n_degs_ensembl

(5860, 3996)

In [25]:
enr.set_db(geneset_num=0)

In [26]:
shortId, userListId = enr.open_session_upload_symbols(enr.degs_ensembl)
shortId, userListId

('103f3ed0bee271fb3b1240a05e258dde', 105346664)

### All enriched cases for many databases

In [27]:
fdr_ptw_cutoff_list = enr.fdr_ptw_cutoff_list
fdr_ptw_cutoff_list

[np.float64(0.05),
 np.float64(0.1),
 np.float64(0.15),
 np.float64(0.2),
 np.float64(0.25),
 np.float64(0.3),
 np.float64(0.35),
 np.float64(0.4),
 np.float64(0.45),
 np.float64(0.5),
 np.float64(0.55),
 np.float64(0.6),
 np.float64(0.65),
 np.float64(0.7),
 np.float64(0.75)]

In [28]:
# dfsim = pdreadcsv( enr.cfg.fname_lfc_cutoff, enr.cfg.root_config)
dfsim = enr.open_simulation_table()
if dfsim is None:
    dfsim = pd.DataFrame()

print(len(dfsim))
dfsim.tail(3)

5822


,case,normalization,cutoff,abs_lfc_cutoff,fdr_lfc_cutoff,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
5819,G4,not_normalized,0.000 - 0.730,0.0,0.73,"['CA8', 'LINGO3', 'LHX2', 'HCN1', 'SCARNA13', 'GLI1', 'ID2', 'FUT1', 'SLC22A...",18624,"['CA8', 'LINGO3', 'LHX2', 'HCN1', 'SCARNA13', 'GLI1', 'ID2', 'FUT1', 'SLC22A...",11448,"['A1CF', 'AAA1', 'AAAS', 'AACS', 'AADAC', 'AADACL3', 'AADAT', 'AAGAB', 'AAK1...",10222,"['A1CF', 'AAAS', 'AACS', 'AADAC', 'AADACL3', 'AADAT', 'AAGAB', 'AAK1', 'AAMP...",5873,"['A1BG', 'A1BG-AS1', 'A2M', 'A2ML1', 'A4GALT', 'AACSP1', 'AANAT', 'AASDHPPT'...",8402,"['A1BG', 'A1BG-AS1', 'A2M', 'A2ML1', 'A4GALT', 'AACSP1', 'AANAT', 'AASDHPPT'...",5575
5820,G4,not_normalized,0.000 - 0.740,0.0,0.74,"['CA8', 'LHX2', 'HCN1', 'LINGO3', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'SLC22A...",18955,"['CA8', 'LHX2', 'HCN1', 'LINGO3', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'SLC22A...",11635,"['A1CF', 'AAA1', 'AAAS', 'AACS', 'AADAC', 'AADACL3', 'AADACL4', 'AADAT', 'AA...",10416,"['A1CF', 'AAAS', 'AACS', 'AADAC', 'AADACL3', 'AADACL4', 'AADAT', 'AAGAB', 'A...",5978,"['A1BG', 'A1BG-AS1', 'A2M', 'A2ML1', 'A4GALT', 'AACSP1', 'AANAT', 'AASDHPPT'...",8539,"['A1BG', 'A1BG-AS1', 'A2M', 'A2ML1', 'A4GALT', 'AACSP1', 'AANAT', 'AASDHPPT'...",5657
5821,G4,not_normalized,0.000 - 0.750,0.0,0.75,"['LHX2', 'LINGO3', 'HCN1', 'CA8', 'GLI1', 'ID2', 'SCARNA13', 'FUT1', 'SLC22A...",19255,"['LHX2', 'LINGO3', 'HCN1', 'CA8', 'GLI1', 'ID2', 'SCARNA13', 'FUT1', 'SLC22A...",11797,"['A1CF', 'AAA1', 'AAAS', 'AACS', 'AADAC', 'AADACL3', 'AADACL4', 'AADAT', 'AA...",10567,"['A1CF', 'AAAS', 'AACS', 'AADAC', 'AADACL3', 'AADACL4', 'AADAT', 'AAGAB', 'A...",6060,"['A1BG', 'A1BG-AS1', 'A2M', 'A2ML1', 'A4GALT', 'AACSP1', 'AANAT', 'AASDHPPT'...",8688,"['A1BG', 'A1BG-AS1', 'A2M', 'A2ML1', 'A4GALT', 'AACSP1', 'AANAT', 'AASDHPPT'...",5737


In [29]:
enr.lfc_list

array([0.   , 0.025, 0.05 , 0.075, 0.1  , 0.125, 0.15 , 0.175, 0.2  ,
       0.225, 0.25 , 0.275, 0.3  , 0.325, 0.35 , 0.375, 0.4  , 0.425,
       0.45 , 0.475, 0.5  , 0.525, 0.55 , 0.575, 0.6  , 0.625, 0.65 ,
       0.675, 0.7  , 0.725, 0.75 , 0.775, 0.8  , 0.825, 0.85 , 0.875,
       0.9  , 0.925, 0.95 , 0.975, 1.   ])

In [30]:
enr.fdr_list

array([0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11, 0.12, 0.13, 0.14, 0.15,
       0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22, 0.23, 0.24, 0.25, 0.26,
       0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37,
       0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48,
       0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59,
       0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7 ,
       0.71, 0.72, 0.73, 0.74, 0.75])

### How many samples per case?

In [31]:
for case in case_list:
    dfsim2 = dfsim[ (dfsim.case == case) & (dfsim.normalization == enr.normalization) & 
                    (dfsim.n_degs >= enr.num_min_degs_for_ptw_enr)]
    print(f"case {case} #simulations {len(dfsim2)}")

case WNT #simulations 2911
case G4 #simulations 2911


In [32]:
dfsim2 = dfsim[ (dfsim.normalization == enr.normalization) & (dfsim.n_degs > 2)].copy()
dfsim2.index = np.arange(0, len(dfsim2))
print(len(dfsim2))

5822


In [33]:
for i in range(len(dfsim2)):
    row = dfsim2.iloc[i]
    degs = eval(row.degs)

    case = row.case
    abs_lfc_cutoff = row.abs_lfc_cutoff
    fdr_lfc_cutoff = row.fdr_lfc_cutoff

    print(i, case, abs_lfc_cutoff, fdr_lfc_cutoff, len(degs), degs[:9], '...')
    if i > 3: break

0 WNT 1.0 0.05 268 ['SYN2', 'HOXD8', 'FOXA1', 'SFRP1', 'LRIT2', 'LOC283392', 'ARPP21', 'lncRNA34', 'CSGALNACT1'] ...
1 WNT 1.0 0.06 397 ['SYN2', 'HOXD8', 'FOXA1', 'RGS7BP', 'SFRP1', 'XLOC_002408', 'LOC283392', 'lncRNA2', 'LRIT2'] ...
2 WNT 1.0 0.07 479 ['SYN2', 'HOXD8', 'FOXA1', 'TMEM233', 'RUNX2', 'FSTL5', 'NOL4', 'VWDE', 'SLC8A1'] ...
3 WNT 1.0 0.08 567 ['SYN2', 'HOXD8', 'FOXA1', 'RGS7BP', 'XLOC_002408', 'VSTM2L', 'lncRNA2', 'lncRNA13', 'RUNX2'] ...
4 WNT 1.0 0.09 688 ['SYN2', 'HOXD8', 'FOXA1', 'NKD1', 'SLC22A15', 'TMEM163', 'RNF175', 'RGS7BP', 'lncRNA12'] ...


In [34]:
enr.abs_lfc_cutoff_inf

0.4

In [35]:
dfsim[ (dfsim.case == 'WNT') & (dfsim.abs_lfc_cutoff == abs_lfc_cutoff_inf) & (dfsim.fdr_lfc_cutoff == 0.15)]

,case,normalization,cutoff,abs_lfc_cutoff,fdr_lfc_cutoff,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
1714,WNT,not_normalized,0.400 - 0.150,0.4,0.15,"['SYN2', 'HOXD8', 'FOXA1', 'VSTM2L', 'FAM155A', 'lncRNA12', 'TMEM233', 'KCNK...",1227,"['SYN2', 'HOXD8', 'FOXA1', 'VSTM2L', 'TMEM233', 'KCNK10', 'NOL4', 'VWDE', 'K...",890,"['AANAT', 'ABCA1', 'ABHD14B', 'ACCS', 'ACTR3B', 'ADAM19', 'ADAMTSL1', 'AMHR2...",413,"['AANAT', 'ABCA1', 'ABHD14B', 'ACCS', 'ACTR3B', 'ADAM19', 'ADAMTSL1', 'AMHR2...",257,"['ABCA7', 'ABHD13', 'ABHD3', 'ACADL', 'ACTL8', 'ADAMTS1', 'ADAMTS17', 'ADAP1...",814,"['ABCA7', 'ABHD13', 'ABHD3', 'ACADL', 'ACTL8', 'ADAMTS1', 'ADAMTS17', 'ADAP1...",633


In [36]:
dfsim[ (dfsim.case == 'G4') & (dfsim.abs_lfc_cutoff == abs_lfc_cutoff_inf) & (dfsim.fdr_lfc_cutoff == 0.15)]

,case,normalization,cutoff,abs_lfc_cutoff,fdr_lfc_cutoff,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
4625,G4,not_normalized,0.400 - 0.150,0.4,0.15,"['LINGO3', 'LHX2', 'CA8', 'HCN1', 'SCARNA13', 'ID2', 'GLI1', 'FUT1', 'VSTM2L...",1757,"['LINGO3', 'LHX2', 'CA8', 'HCN1', 'SCARNA13', 'ID2', 'GLI1', 'FUT1', 'VSTM2L...",1212,"['AACS', 'AATF', 'ABCA1', 'ACRV1', 'ACSM3', 'ADAMTS4', 'AHI1', 'AHNAK2', 'AN...",611,"['AACS', 'AATF', 'ABCA1', 'ACRV1', 'ACSM3', 'ADAMTS4', 'AHI1', 'AHNAK2', 'AN...",322,"['A1BG', 'ABCA5', 'ABHD15', 'ABLIM3', 'ABR', 'ABRA', 'ACADL', 'ACAN', 'ACAP2...",1146,"['A1BG', 'ABCA5', 'ABHD15', 'ABLIM3', 'ABR', 'ABRA', 'ACADL', 'ACAN', 'ACAP2...",890


### Calc all enrichment analyses

In [37]:
geneset_num_list = [1, 2, 4, 5, 7]
geneset_num_list = [0, 1, 2, 4, 5, 7]
geneset_num_list = [0]

In [38]:
enr.set_db(0, verbose=True)

>>> Reactome_Pathways_2024


In [39]:
enr.set_enrichment_name()

('enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv',
 'enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_WNT_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000_pathway_pval_0.050_fdr_0.050_num_genes_0.05.tsv')

In [40]:
enr.abs_lfc_cutoff_inf

0.4

In [41]:
print(enr.abs_lfc_cutoff_inf)
df_fdr = enr.open_fdr_lfc_correlation(case, enr.abs_lfc_cutoff_inf)

0.4


In [42]:
dfsim = enr.open_simulation_table()
dfsim.head(3)

,case,normalization,cutoff,abs_lfc_cutoff,fdr_lfc_cutoff,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
0,WNT,not_normalized,1.000 - 0.050,1.0,0.05,"['SYN2', 'HOXD8', 'FOXA1', 'SFRP1', 'LRIT2', 'LOC283392', 'ARPP21', 'lncRNA3...",268,"['SYN2', 'HOXD8', 'FOXA1', 'SFRP1', 'LRIT2', 'ARPP21', 'CSGALNACT1', 'RASGRP...",211,"['ADAM19', 'BMF', 'C1orf130', 'C2orf71', 'CDHR1', 'DKK2', 'DLX3', 'DLX4', 'D...",75,"['ADAM19', 'BMF', 'CDHR1', 'DKK2', 'DLX3', 'DLX4', 'DSE', 'DSG2', 'DSP', 'EM...",52,"['ADARB2', 'ADD3', 'ALDH1A3', 'ALDH5A1', 'ANKRD34A', 'ANKRD43', 'ANO4', 'ANX...",193,"['ADARB2', 'ADD3', 'ALDH1A3', 'ALDH5A1', 'ANKRD34A', 'ANO4', 'ANXA3', 'ARAP2...",159
1,WNT,not_normalized,1.000 - 0.060,1.0,0.06,"['SYN2', 'HOXD8', 'FOXA1', 'RGS7BP', 'SFRP1', 'XLOC_002408', 'LOC283392', 'l...",397,"['SYN2', 'HOXD8', 'FOXA1', 'RGS7BP', 'SFRP1', 'LRIT2', 'ARPP21', 'RASGRP1', ...",302,"['ABCA1', 'ADAM19', 'ATP6V1C2', 'AXIN2', 'BMF', 'C1orf130', 'C22orf31', 'C2o...",122,"['ABCA1', 'ADAM19', 'ATP6V1C2', 'AXIN2', 'BMF', 'C22orf31', 'CACNA2D3', 'CAM...",81,"['ADARB2', 'ADD3', 'ADORA2B', 'ALDH1A3', 'ALDH5A1', 'ANGPT1', 'ANKRD32', 'AN...",275,"['ADARB2', 'ADD3', 'ADORA2B', 'ALDH1A3', 'ALDH5A1', 'ANGPT1', 'ANKRD34A', 'A...",221
2,WNT,not_normalized,1.000 - 0.070,1.0,0.07,"['SYN2', 'HOXD8', 'FOXA1', 'TMEM233', 'RUNX2', 'FSTL5', 'NOL4', 'VWDE', 'SLC...",479,"['SYN2', 'HOXD8', 'FOXA1', 'TMEM233', 'RUNX2', 'FSTL5', 'NOL4', 'VWDE', 'SLC...",368,"['ABCA1', 'ACTR3B', 'ADAM19', 'ATP6V1C2', 'AXIN2', 'BMF', 'BSG', 'C1orf130',...",148,"['ABCA1', 'ACTR3B', 'ADAM19', 'ATP6V1C2', 'AXIN2', 'BMF', 'BSG', 'C22orf31',...",101,"['ADARB2', 'ADD3', 'ADORA2B', 'ALDH1A3', 'ALDH5A1', 'ANGPT1', 'ANKRD32', 'AN...",331,"['ADARB2', 'ADD3', 'ADORA2B', 'ALDH1A3', 'ALDH5A1', 'ANGPT1', 'ANKRD34A', 'A...",267


### Calc DEFAULT paramenters Enrichment Analysis

In [43]:
force = False
verbose = False
enr.calc_default_enrichment_analysis(geneset_num_list=[0, 1, 2, 4, 5, 7], force=force, verbose=verbose)

### Reactome in Enricher run in 2024-03-10

In [44]:
enr.abs_lfc_cutoff_inf

0.4

In [45]:
case = case_list[1]

abs_lfc_cutoff_inf = 1
abs_lfc_cutoff_inf = enr.abs_lfc_cutoff_inf

df_fdr = enr.open_fdr_lfc_correlation(case, abs_lfc_cutoff_inf)

try:
    df2 = df_fdr[ pd.notnull(df_fdr['corr']) ]
    print(len(df2))
except:
    df2 = pd.DataFrame()

print(len(df2))
df2

11
11


,case,first,chosen,fdr,corr,label,method,n_degs_min,n_degs_max,abs_lfc_cutoff_inf,degs_min,degs_max
8,G4,False,False,0.13,-0.471,fdr=0.13 not found corr.,spearman,1399,1409,0.4,"['LHX2', 'LINGO3', 'CA8', 'HCN1', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'VSTM2L...","['LHX2', 'LINGO3', 'HCN1', 'CA8', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'NAV3',..."
9,G4,False,False,0.14,-0.699,fdr=0.14 not found corr.,spearman,1555,1574,0.4,"['LINGO3', 'HCN1', 'LHX2', 'CA8', 'GLI1', 'SCARNA13', 'ID2', 'FUT1', 'SLC22A...","['LINGO3', 'CA8', 'LHX2', 'HCN1', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'VSTM2L..."
10,G4,False,False,0.15,-0.749,fdr=0.15 not found corr.,spearman,1720,1757,0.4,"['CA8', 'LINGO3', 'HCN1', 'LHX2', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'NAV3',...","['LINGO3', 'LHX2', 'CA8', 'HCN1', 'SCARNA13', 'ID2', 'GLI1', 'FUT1', 'VSTM2L..."
11,G4,False,False,0.16,-0.792,fdr=0.16 not found corr.,spearman,1849,1913,0.4,"['LHX2', 'HCN1', 'LINGO3', 'CA8', 'SCARNA13', 'ID2', 'GLI1', 'FUT1', 'NAV3',...","['LHX2', 'HCN1', 'CA8', 'LINGO3', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'SLC22A..."
12,G4,False,False,0.17,-0.859,fdr=0.17 not found corr.,spearman,1977,2079,0.4,"['HCN1', 'LHX2', 'CA8', 'LINGO3', 'GLI1', 'ID2', 'SCARNA13', 'FUT1', 'NAV3',...","['LHX2', 'LINGO3', 'CA8', 'HCN1', 'GLI1', 'SCARNA13', 'ID2', 'FUT1', 'VSTM2L..."
13,G4,False,False,0.18,-0.886,fdr=0.18 not found corr.,spearman,2138,2295,0.4,"['LINGO3', 'HCN1', 'CA8', 'LHX2', 'GLI1', 'SCARNA13', 'ID2', 'FUT1', 'SLC22A...","['LHX2', 'LINGO3', 'HCN1', 'CA8', 'SCARNA13', 'ID2', 'GLI1', 'FUT1', 'VSTM2L..."
14,G4,True,True,0.19,-0.908,fdr=0.19 corr=-0.908 ***,spearman,2315,2527,0.4,"['CA8', 'HCN1', 'LINGO3', 'LHX2', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'SLC22A...","['CA8', 'LINGO3', 'LHX2', 'HCN1', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'SLC22A..."
15,G4,False,True,0.2,-0.927,fdr=0.20 corr=-0.927,spearman,2465,2760,0.4,"['LINGO3', 'LHX2', 'CA8', 'HCN1', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'VSTM2L...","['CA8', 'LHX2', 'LINGO3', 'HCN1', 'GLI1', 'SCARNA13', 'ID2', 'FUT1', 'NAV3',..."
16,G4,False,True,0.21,-0.943,fdr=0.21 corr=-0.943,spearman,2616,3017,0.4,"['LHX2', 'CA8', 'HCN1', 'LINGO3', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'NAV3',...","['LINGO3', 'LHX2', 'HCN1', 'CA8', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'VSTM2L..."
17,G4,False,True,0.22,-0.957,fdr=0.22 corr=-0.957,spearman,2775,3286,0.4,"['HCN1', 'LHX2', 'LINGO3', 'CA8', 'GLI1', 'SCARNA13', 'ID2', 'FUT1', 'SLC22A...","['LHX2', 'LINGO3', 'HCN1', 'CA8', 'ID2', 'GLI1', 'SCARNA13', 'FUT1', 'VSTM2L..."


In [46]:
df_fdr = enr.open_fdr_lfc_correlation(case, enr.abs_lfc_cutoff_inf)
df_fdr.head(2)

,case,first,chosen,fdr,corr,label,method,n_degs_min,n_degs_max,abs_lfc_cutoff_inf,degs_min,degs_max
0,G4,False,False,0.05,NaN,fdr=0.05 not found corr.,spearman,206,206,0.4,"['CA8', 'LHX2', 'LINGO3', 'HCN1', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'SLC22A...","['CA8', 'LHX2', 'LINGO3', 'HCN1', 'ID2', 'SCARNA13', 'GLI1', 'FUT1', 'SLC22A..."
1,G4,False,False,0.06,NaN,fdr=0.06 not found corr.,spearman,318,318,0.4,"['LINGO3', 'HCN1', 'LHX2', 'CA8', 'GLI1', 'ID2', 'SCARNA13', 'FUT1', 'SLC22A...","['LINGO3', 'HCN1', 'LHX2', 'CA8', 'GLI1', 'ID2', 'SCARNA13', 'FUT1', 'SLC22A..."


In [47]:
verbose = False
geneset_num_list = [0]
# remove the comments - it last some minutes
enr.calc_all_enrichment_analysis(geneset_num_list, iecho=50, force=force, verbose=verbose)

# WNT: 725 samples; abs_lfc_cutoff >= 0.4 and DEGs >= 3

geneset = Reactome_Pathways_2024

0 / 725) DEGs 211 for WNT, lfc cutoff = 1.0 and fdr cutoff = 0.05
For case 0 'WNT' ('WNT'), there are 268/211 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=1.000; FDR=0.050

.............................
50 / 725) DEGs 1757 for WNT, lfc cutoff = 0.975 and fdr cutoff = 0.26
For case 0 'WNT' ('WNT'), there are 2476/1757 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.975; FDR=0.260

.......................
100 / 725) DEGs 1147 for WNT, lfc cutoff = 0.925 and fdr cutoff = 0.18
For case 0 'WNT' ('WNT'), there are 1591/1147 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.925; FDR=0.180

...............................................................
200 / 725) DEGs 2249 for WNT, lfc cutoff = 0.85 and fdr cutoff = 0.31
For case 0 'WNT' ('WNT'), there are 3170/2249 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.850; FDR=0.310

...................
250 / 725) DEGs 1610 for WNT, lfc cutoff

In [49]:
verbose = False
geneset_num_list = [1, 2, 4, 5, 7]
# remove the comments to run - it lasts some minutes
enr.calc_all_enrichment_analysis(geneset_num_list, iecho=50, force=force, verbose=verbose)

# WNT: 725 samples; abs_lfc_cutoff >= 0.4 and DEGs >= 3

geneset = WikiPathways_2024_Human

0 / 725) DEGs 211 for WNT, lfc cutoff = 1.0 and fdr cutoff = 0.05
For case 0 'WNT' ('WNT'), there are 268/211 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=1.000; FDR=0.050

....................................
50 / 725) DEGs 1757 for WNT, lfc cutoff = 0.975 and fdr cutoff = 0.26
For case 0 'WNT' ('WNT'), there are 2476/1757 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.975; FDR=0.260

........................
100 / 725) DEGs 1147 for WNT, lfc cutoff = 0.925 and fdr cutoff = 0.18
For case 0 'WNT' ('WNT'), there are 1591/1147 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.925; FDR=0.180

...........................................................
200 / 725) DEGs 2249 for WNT, lfc cutoff = 0.85 and fdr cutoff = 0.31
For case 0 'WNT' ('WNT'), there are 3170/2249 DEGs/DEGs with ensembl_id
DEG's cutoffs: abs(LFC)=0.850; FDR=0.310

.....................
250 / 725) DEGs 1610 for WNT, lfc

### Sampling Pathways 

In [ ]:
dfa = enr.count_sampling(geneset_num_list=[0], prompt_verbose=True)
len(dfa)

In [ ]:
fig, dfa = enr.barplot_sampling_cutoffs(prompt_verbose=False, verbose=False)
fig.show()

### Differences between databases
#### Run only if you defined teh best config: new05 algorithm

In [ ]:
enr.get_best_ptw_cutoff_biopax()

In [ ]:
case = case_list[0]

In [ ]:
enr.abs_lfc_cutoff , enr.fdr_lfc_cutoff, enr.pathway_pval_cutoff, enr.pathway_fdr_cutoff, enr.num_of_genes_cutoff

In [ ]:
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
len(degs)

In [ ]:
fname, fname_cutoff = enr.set_enrichment_name()
fname, fname_cutoff 

In [ ]:
geneset_num_list = [0, 1, 2, 4, 5, 7]

for geneset_num in geneset_num_list:
    enr.set_db(geneset_num, verbose=True)

### Reactome_2022

In [ ]:
enr.set_db(0, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
# print("\nEcho Parameters:")
enr.echo_parameters()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()
print(len(enr.df_enr))
enr.df_enr

### Reactome_2022 case G4

In [ ]:
enr.set_db(0, verbose=True)
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
# print("\nEcho Parameters:")
enr.echo_parameters()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()
print(len(enr.df_enr))
enr.df_enr

### WikiPathway_2021_Human

In [ ]:
enr.set_db(1, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)

In [ ]:
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(42)

In [ ]:
enr.df_enr.tail(40)

### KEGG

In [ ]:
enr.set_db(2, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(45)

In [ ]:
enr.df_enr.tail(40)

In [ ]:
enr.set_db(2, verbose=True)
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

enr.df_enr.head(30)

In [ ]:
enr.df_enr.tail(30)

### BioPlanet_2019 = 4

In [ ]:
enr.set_db(4, verbose=True)
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(57)

In [ ]:
enr.df_enr.tail(50)

In [ ]:
enr.set_db(4, verbose=True)
case = case_list[1]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
enr.echo_enriched_pathways()

In [ ]:
if enr.df_enr is None:
    enr.df_enr = pd.DataFrame()

print(len(enr.df_enr))
enr.df_enr.head(50)

In [ ]:
enr.df_enr.tail(43)

### Development & tests

In [ ]:
force=True; verbose=False
num_min_degs_for_ptw_enr = 3

geneset_num_list = [1, 2, 4, 5, 7]
geneset_num_list = [0, 1, 2, 4, 5, 7]
geneset_num_list = [0]

icount=-1
for case in case_list:
    if not enr.open_case_simple(case):
        print(f"Problems for {case} !!!!")
        continue
    
    dfsim2 = dfsim[ (dfsim.normalization == enr.normalization) & (dfsim.case == case) &
                    (dfsim.n_degs >= num_min_degs_for_ptw_enr)]
    
    for i in range(len(dfsim2)):
        icount += 1
        
        row = dfsim2.iloc[i]

        degs = eval(row.degs)
        case = row.case
        
        abs_lfc_cutoff = row.abs_lfc_cutoff
        fdr_lfc_cutoff = row.fdr_lfc_cutoff

        degs2, _ = enr.list_of_degs_params(abs_lfc_cutoff, fdr_lfc_cutoff, verbose=False)

        if len(degs) != len(degs2):
            print("Error:", case, abs_lfc_cutoff, fdr_lfc_cutoff, len(degs), len(degs2))
            continue

        # if i > 10:break
        enr.calc_EA_dataset_symbol(degs, return_value=True, force=force, verbose=verbose)
        if icount%100==0:
            print(case, len(degs), abs_lfc_cutoff, fdr_lfc_cutoff)
            enr.echo_degs()
            print("")
            enr.echo_enriched_pathways()
            print("\n")


### Reactome, Bioplanet, KEGG

In [ ]:
enr.dbs_list

In [ ]:
[enr.dbs_list[i] for i in [0, 1, 2, 4, 5, 7]]

In [ ]:
enr.set_which_db('xxx')

In [ ]:
enr.set_which_db('Reactome_2022')

In [ ]:
enr.set_which_db('Reactome')

In [ ]:
enr.set_which_db('reactome')

In [ ]:
enr.set_which_db('KEGG_2021')

In [ ]:
enr.set_which_db('KEGG')

In [ ]:
enr.set_db(geneset_num = 0)

### Start with Reactome

In [ ]:
case_list

In [ ]:
case = case_list[0]
enr.open_case(case, verbose=False)
degs, dfdegs = enr.list_of_degs(force=False, verbose=True)

# enr.set_which_db('Reactome_2022')
geneset_num = 0
force=False; verbose=False

print(f"case: {case}")
print(f"there are {len(degs)} for fdr < {enr.fdr_lfc_cutoff:.3f} and lfc >= {enr.abs_lfc_cutoff:.3f}")
print(f"Pathway filter fdr < {enr.fdr_pathway_cutoff:.3f} and lfc >= {enr.pval_pathway_cutoff:.3f} and num_of_genes_cutoff={enr.num_of_genes_cutoff}")

enr.calc_EA_dataset_symbol(degs, force=force, verbose=verbose)

In [ ]:
i=0
case = case_list[i]
ret, degs, dfdegs = enr.open_case(case, verbose=False)

enr.df_enr

In [ ]:
print(enr.geneset_lib)

row = enr.get_enriched_pathway_line(i_line=0)

if row is not None:
    print(row.pathway, row.pathway_id)
    print(len(enr.genes_in_pathway), enr.genes_in_pathway)


### Enriched DEGs

In [ ]:
len(enr.all_enr_degs), ",".join(enr.all_enr_degs)

In [ ]:
",".join(enr.degs)

In [ ]:
enr_found_degs = [x for x in enr.degs if x in enr.all_enr_degs]
",".join(enr_found_degs)

### Found genes 

In [ ]:
len(enr.all_enr_degs), ",".join(enr.all_enr_degs)

In [ ]:
len(enr.enr_found_degs), ",".join(enr.enr_found_degs)

### Not found genes in pathways

In [ ]:
len(enr.enr_not_found_degs), ",".join(enr.enr_not_found_degs)

### BioPlanet

In [ ]:
enr.set_db(geneset_num=4)

In [ ]:
dfbiop = enr.open_db_pathway()
print(len(dfbiop))
dfbiop.head(3)

### Bioplanet diseases

In [ ]:
_ = enr.open_bioplanet_disease(force=False)
dfdisease = enr.dfdisease
print(len(dfdisease))
dfdisease.head()

### Bioplanet Categories

In [ ]:
 _ = enr.open_bioplanet_category(force=False)
dfbiop_cat = enr.dfbiop_cat
print(type(dfbiop_cat.category))
print(len(dfbiop_cat))
dfbiop_cat.head()

In [ ]:
def all_equal_list(cols1, cols2):
    if cols1 is None and cols2 is None: return True
    if cols1 == [] and cols2 == []: return True
    
    if len(cols1) != len(cols2): return False
    
    soma = np.sum([1 if cols1[i] != cols2[i] else 0 for i in range(len(cols1))])
    return soma == 0

cols1 = list(enr.dflfc_all.columns)

cols2 = ['probe', 'symbol', 'geneid', 'description', 'logFC', 'meanExpr', \
        't.stat', 'p-value', 'fdr', 'B', 'chr.range', 'org.chromosome', \
        'forward.reverse', 'nuc.sequence', 'gemmaid', 'go.term']

all_equal_list(cols1, cols2)

In [ ]:
all_genes = []
for i in range(len(dfsig)):
    genes = eval(dfsig.iloc[i].genes)
    # print(i, len(genes))
    all_genes += genes
    
all_genes = np.unique(all_genes)
all_genes.sort()
all_genes

not_found_genes = np.array([x for x in enr.deg_list if not x in all_genes])
not_found_genes

In [ ]:
def find_hugo_symbol(gene):
    try:
        i = enr.gene.dic_genes[gene]
        if isinstance(i, list):
            i = i[0]
            
        mat = enr.gene.df_synonyms.iloc[i]['synonyms']
        print(">>>", gene, i, mat, type(mat))
        if isinstance(mat, str):
            mat = eval(mat)
            
        gene0 = mat[0]
    except:
        gene0 = gene

    return gene0

In [ ]:
gene = 'SEPP1'
find_hugo_symbol(gene)

In [ ]:
enr.dic_genes[gene]

In [ ]:
lista = [x for x in os.listdir(enr.root_result) if 'taubate_PAC_UP_' in x and not '~lock' in x ]
print(len(lista))
", ".join(lista)


In [ ]:
root = enr.root_result
_type = '.tsv'
_type = None
pattern_src = 'taubate_PAC'
pattern_dst = 'taubate_MAP'

if _type is None:
    lista = [x for x in os.listdir(root) if pattern_src in x and not '~lock' in x ]
else:
    lista = [x for x in os.listdir(root) if pattern_src in x and not '~lock' in x and x.endswith(_type)]

if lista == []:
    print(f"There are no files in {root}")

for fname in lista:
    fname_src = os.path.join(root, fname)

    fname_dst = fname.replace(pattern_src, pattern_dst)
    fname_dst = os.path.join(root, fname_dst)

    if not  os.path.exists(fname_src):
        print(f"fname does not exist: '{fname_src}'", fname_src)
        continue
    if os.path.exists(fname_dst):
        print(f"fname already exists: '{fname_dst}'")
        continue
        
    if fname_dst == fname_src:
        print(f"fname did not change: 'fname_src'")
        continue
        
    print(fname_src, 'to', fname_dst)
    os.rename(fname_src, fname_dst)

<p style="font-size: 24px; color: cyan;">
$Hypergeometric Probability = p(k,M, n, N) = \frac{\binom{n}{k} * \binom{M-n}{N-k}}{\binom{M}{N}}$
</p>

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html

### The Super-Hypergeometric statistics
  - given all enriched pathways
  - given all DEGs = n
  - given all degs in pathways = k

  - given all genes declared in the experiment (microarray for example) - n_total_genes = M
  - given all genes annotated in the enriched pathways (n_annotated) = N

  - what is the possibility to find n_degs_in_pathays given that M, n, N




In [ ]:
from scipy.stats import hypergeom

In [ ]:
n = 900 # DEGs
M = 2000 # Microarray
N = 300  # annottated in pathways
k = 150  # degs in pathways

In [ ]:
p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

p

### Testing

In [ ]:
case  = case_list[0]
enr.open_case(case, verbose=False)

df_enr = enr.merge_reactome(enr.df_enr)
df_enr.head()

<p style="font-size: 24px; color: cyan;">
$Hypergeometric Probability = p(k,M, n, N) = \frac{\binom{n}{k} * \binom{M-n}{N-k}}{\binom{M}{N}}$
</p>

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html

In [ ]:
n = enr.n_degs # DEGs
M = enr.n_total_genes # Microarray

if enr.calc_all_annoted_genes():
    N = enr.n_annotated_genes # annottated in pathways
else:
    N = 0
    
k = enr.n_degs_in_pathways  # degs in pathways

print(f"There are {N} genes annotated in pathways and {M} genes available in the microarray experiment")
print(f"We found {k} DEGs in the enriched pathways in {n} DEGs obtained by new algorithm and cutoffs")

p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

if p < 0.05:
    print(f"We reject H0, is statistically unlikely to find {k} DEGs in the pathways, p-value = {p:.3e}")
else:
    print(f"We accept H0, is statistically likely to find {k} DEGs in the pathways, p-value = {p:.3e}")
    

In [ ]:
n = 3   # pathway having {n} word
M = 40000 # total words
N = 30 # pathways' words in teh bag
k = 1  # degs in pathways

print(f"Tere are {N} pathways' words in the bag of {M} words in MB - WNT")
print("")

for pathway, k, n in [('path1', 1, 2), ('path2', 2, 5), ('path3', 3, 8)]:
    print(f"For pathway '{pathway}', we found {k} word(s) in the pathway having {n} words")
    
    p = hypergeom.cdf(k, M, n, N)
    if p >= .9:
        p = 1 - p
    else:
        p
    
    if p < 0.05:
        print(f"We reject H0, is statistically unlikely to find {k} DEGs in the pathways, p-value = {p:.3e}")
    else:
        print(f"We accept H0, is statistically likely to find {k} DEGs in the pathways, p-value = {p:.3e}")

    print("")

### Testing curation
#### first WNT

In [ ]:
n = 2100 # papers in Medulloblastom
M = 48151 # Pubmed Papers related to Neuronal Cancer 
N = 38  # pathways related to WNT Medulloblastoma, the come from the Literature
k = 23  # pathways for WNT found by curations

case2 = 'WNT'
print(f"For {case2}")
print(f"There are {N} enriched pathways in {case2} Medulloblastoma and {M} Pubmed Papers related to Neuronal Cancer ")
print(f"We found {k} pathways for {case2} found by curations in {n} papers in Medulloblastom")

p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

if p < 0.05:
    print(f"We reject H0, is statistically unlikely to curate {k} pathways, p-value = {p:.3e}")
else:
    print(f"We accept H0, is statistically likely to curate {k} pathways, p-value = {p:.3e}")


In [ ]:
k = 12  # pathays for G4 found by curations

case2 = 'G4'
print(f"For {case2}")
print(f"There are {N} enriched pathways in {case2} Medulloblastoma and {M} Pubmed Papers related to Neuronal Cancer ")
print(f"We found {k} pathways for {case2} found by curations in {n} papers in Medulloblastom")

p = hypergeom.cdf(k, M, n, N)
if p >= .9:
    p = 1 - p
else:
    p

if p < 0.05:
    print(f"We reject H0, is statistically unlikely to curate {k} pathways, p-value = {p:.3e}")
else:
    print(f"We accept H0, is statistically likely to curate {k} pathways, p-value = {p:.3e}")


In [ ]:
def binomial_find_nDEGs_in_pathways():
    from scipy.stats import binom

p = 0.5
n = len(df2)
x = df2['found'].sum()
x, n


prob = binom.cdf(x, n, p)

if prob <= 0.5:
    pval = prob
        
    if pval < 0.05:
        print(f"Alternative Hypothesis: this finding could shows that the algoritm did a BAD JOB: pval = {pval:.3f}")
    else:
        print(f"Null Hypothesis: this finding could be achieved by random: pval = {pval:.3f}")
else:
    pval = 1 - prob
    
    if pval < 0.05:
        print(f"Alternative Hypothesis: this finding could not be achieved by random: pval = {pval:.3f}")
    else:
        print(f"Null Hypothesis: this finding could be achieved by random: pval = {pval:.3f}")

print(f"The curation found {x} pathways/concepts in {n} possible given pathways")
